In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password="ganesha@03",
    host="localhost",
    port="5432",
    database="finguard_db",
)
engine = create_engine(url)

# Load our engineered feature set (not the raw transactions table)
df = pd.read_sql("SELECT * FROM transactions_features", engine)
print(df.shape)
df.head()

(284807, 36)


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V27,V28,Amount,Class,Hour,Is_High_Risk_Hour,Amount_Log,Amount_Zscore,Is_Round_Amount,Txn_Count_Last_Hour
0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,...,0.133558,-0.021053,149.62,0,0.0,0,5.014760,0.244964,0,0
1,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,...,-0.008983,0.014724,2.69,0,0.0,0,1.305626,-0.342474,0,0
2,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,-0.055353,-0.059752,378.66,0,0.0,0,5.939276,1.160684,0,2
3,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,0.062723,0.061458,123.50,0,0.0,0,4.824306,0.140534,0,2
4,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,...,0.219422,0.215153,69.99,0,0.0,0,4.262539,-0.073403,0,4


In [2]:
# Data is already in chronological order (we sorted by Time in Week 3)
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"\nTrain fraud rate: {train_df['Class'].mean()*100:.4f}%")
print(f"Test fraud rate: {test_df['Class'].mean()*100:.4f}%")

Train shape: (227845, 36)
Test shape: (56962, 36)

Train fraud rate: 0.1830%
Test fraud rate: 0.1317%


In [3]:
# Drop Class (target) and Amount (raw, unscaled - we use Amount_Log instead) from features
feature_cols = [c for c in df.columns if c not in ['Class', 'Amount']]

X_train = train_df[feature_cols]
y_train = train_df['Class']

X_test = test_df[feature_cols]
y_test = test_df['Class']

print(f"Features used ({len(feature_cols)}): {feature_cols}")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

Features used (34): ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Hour', 'Is_High_Risk_Hour', 'Amount_Log', 'Amount_Zscore', 'Is_Round_Amount', 'Txn_Count_Last_Hour']
X_train: (227845, 34), y_train: (227845,)
X_test: (56962, 34), y_test: (56962,)


In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training data, then apply to both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")
print(f"X_train_scaled shape: {X_train_scaled.shape}")

Scaling complete.
X_train_scaled shape: (227845, 34)


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    precision_recall_curve, auc, roc_auc_score
)

# class_weight='balanced' handles imbalance by penalizing misclassifying the minority (fraud) class more
log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_test_scaled)
y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

print("=== Logistic Regression — Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))

print("=== Confusion Matrix ===")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# PR-AUC (the metric that actually matters here, not accuracy)
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
pr_auc = auc(recall, precision)
print(f"\nPR-AUC: {pr_auc:.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

=== Logistic Regression — Classification Report ===
              precision    recall  f1-score   support

       Legit       1.00      0.95      0.98     56887
       Fraud       0.03      0.93      0.05        75

    accuracy                           0.95     56962
   macro avg       0.51      0.94      0.51     56962
weighted avg       1.00      0.95      0.97     56962

=== Confusion Matrix ===
[[54237  2650]
 [    5    70]]

PR-AUC: 0.7686
ROC-AUC: 0.9879


In [6]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(class_weight='balanced', max_depth=8, random_state=42)
dt.fit(X_train_scaled, y_train)

y_pred_dt = dt.predict(X_test_scaled)
y_pred_proba_dt = dt.predict_proba(X_test_scaled)[:, 1]

print("=== Decision Tree — Classification Report ===")
print(classification_report(y_test, y_pred_dt, target_names=['Legit', 'Fraud']))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_dt))

precision_dt, recall_dt, _ = precision_recall_curve(y_test, y_pred_proba_dt)
pr_auc_dt = auc(recall_dt, precision_dt)
print(f"\nPR-AUC: {pr_auc_dt:.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_dt):.4f}")

=== Decision Tree — Classification Report ===
              precision    recall  f1-score   support

       Legit       1.00      0.98      0.99     56887
       Fraud       0.06      0.79      0.11        75

    accuracy                           0.98     56962
   macro avg       0.53      0.88      0.55     56962
weighted avg       1.00      0.98      0.99     56962

=== Confusion Matrix ===
[[55918   969]
 [   16    59]]

PR-AUC: 0.2251
ROC-AUC: 0.8968


In [7]:
from imblearn.over_sampling import SMOTE

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")

smote = SMOTE(random_state=42, sampling_strategy=0.1)  # brings fraud to 10% of majority class, not full 50/50 (avoids overcorrecting)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f"After SMOTE: {pd.Series(y_train_smote).value_counts().to_dict()}")

Before SMOTE: {0: 227428, 1: 417}
After SMOTE: {0: 227428, 1: 22742}


In [8]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train_smote, y_train_smote)

y_pred_rf = rf.predict(X_test_scaled)
y_pred_proba_rf = rf.predict_proba(X_test_scaled)[:, 1]

print("=== Random Forest — Classification Report ===")
print(classification_report(y_test, y_pred_rf, target_names=['Legit', 'Fraud']))
print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_rf))

precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_pred_proba_rf)
pr_auc_rf = auc(recall_rf, precision_rf)
print(f"\nPR-AUC: {pr_auc_rf:.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_rf):.4f}")

=== Random Forest — Classification Report ===
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56887
       Fraud       0.61      0.79      0.69        75

    accuracy                           1.00     56962
   macro avg       0.81      0.89      0.84     56962
weighted avg       1.00      1.00      1.00     56962

=== Confusion Matrix ===
[[56850    37]
 [   16    59]]

PR-AUC: 0.8101
ROC-AUC: 0.9838


In [9]:
from xgboost import XGBClassifier

# scale_pos_weight approximates class balance without needing full SMOTE reliance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train_smote, y_train_smote)

y_pred_xgb = xgb.predict(X_test_scaled)
y_pred_proba_xgb = xgb.predict_proba(X_test_scaled)[:, 1]

print("=== XGBoost — Classification Report ===")
print(classification_report(y_test, y_pred_xgb, target_names=['Legit', 'Fraud']))
print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_xgb))

precision_xgb, recall_xgb, _ = precision_recall_curve(y_test, y_pred_proba_xgb)
pr_auc_xgb = auc(recall_xgb, precision_xgb)
print(f"\nPR-AUC: {pr_auc_xgb:.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_xgb):.4f}")

=== XGBoost — Classification Report ===
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56887
       Fraud       0.30      0.77      0.43        75

    accuracy                           1.00     56962
   macro avg       0.65      0.89      0.72     56962
weighted avg       1.00      1.00      1.00     56962

=== Confusion Matrix ===
[[56752   135]
 [   17    58]]

PR-AUC: 0.7661
ROC-AUC: 0.9573


In [10]:
xgb_v2 = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)
# Train on ORIGINAL scaled data, not SMOTE-resampled — let scale_pos_weight alone handle imbalance
xgb_v2.fit(X_train_scaled, y_train)

y_pred_xgb2 = xgb_v2.predict(X_test_scaled)
y_pred_proba_xgb2 = xgb_v2.predict_proba(X_test_scaled)[:, 1]

print("=== XGBoost (no SMOTE, scale_pos_weight only) ===")
print(classification_report(y_test, y_pred_xgb2, target_names=['Legit', 'Fraud']))
print(confusion_matrix(y_test, y_pred_xgb2))

precision_xgb2, recall_xgb2, _ = precision_recall_curve(y_test, y_pred_proba_xgb2)
pr_auc_xgb2 = auc(recall_xgb2, precision_xgb2)
print(f"\nPR-AUC: {pr_auc_xgb2:.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_xgb2):.4f}")

=== XGBoost (no SMOTE, scale_pos_weight only) ===
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56887
       Fraud       0.87      0.73      0.80        75

    accuracy                           1.00     56962
   macro avg       0.94      0.87      0.90     56962
weighted avg       1.00      1.00      1.00     56962

[[56879     8]
 [   20    55]]

PR-AUC: 0.7894
ROC-AUC: 0.9839


In [11]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm.fit(X_train_scaled, y_train)

y_pred_lgbm = lgbm.predict(X_test_scaled)
y_pred_proba_lgbm = lgbm.predict_proba(X_test_scaled)[:, 1]

print("=== LightGBM — Classification Report ===")
print(classification_report(y_test, y_pred_lgbm, target_names=['Legit', 'Fraud']))
print(confusion_matrix(y_test, y_pred_lgbm))

precision_lgbm, recall_lgbm, _ = precision_recall_curve(y_test, y_pred_proba_lgbm)
pr_auc_lgbm = auc(recall_lgbm, precision_lgbm)
print(f"\nPR-AUC: {pr_auc_lgbm:.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_lgbm):.4f}")

=== LightGBM — Classification Report ===
              precision    recall  f1-score   support

       Legit       1.00      0.98      0.99     56887
       Fraud       0.04      0.77      0.08        75

    accuracy                           0.98     56962
   macro avg       0.52      0.87      0.53     56962
weighted avg       1.00      0.98      0.99     56962

[[55552  1335]
 [   17    58]]

PR-AUC: 0.4078
ROC-AUC: 0.8590
